# 🧠 تدريب موديل تصنيف أورام المخ (Brain Tumor MRI)

**الخطوات (من غير ما تفهم أي كود):**
1. من القائمة فوق: **Runtime ← Change runtime type ← اختار T4 GPU ← Save** (ده بيخلي التدريب أسرع).
2. من القائمة: **Runtime ← Run all** (أو دوس ▶️ على كل خلية بالترتيب من فوق لتحت).
3. استنى لحد ما تخلص (حوالي 15–30 دقيقة). في الآخر هينزل لك تلقائيًا ملف `brain_tumor_results.zip`.
4. فك الضغط وحط `brain_tumor_model.keras` و `class_names.json` في فولدر `model` بتاع المشروع.
5. خد أرقام الدقة (Accuracy) والصور (confusion matrix و training curves) وحطهم في التقرير.

> لو ظهرت رسالة تطلب تسجيل دخول Kaggle في الخلية التالتة: اتبع التعليمات اللي بتظهر (محتاج حساب Kaggle مجاني).

In [ ]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## 1) تحميل الداتا من Kaggle

In [ ]:
import kagglehub

try:
    path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")
except Exception as e:
    print("محتاج تسجيل دخول Kaggle:", e)
    kagglehub.login()   # هيظهر لك مربع تحط فيه اسم المستخدم والـ API key
    path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")

print("Dataset path:", path)

In [ ]:
import os

def find_dir(root, name):
    for r, dirs, _ in os.walk(root):
        for d in dirs:
            if d.lower() == name.lower():
                return os.path.join(r, d)
    raise FileNotFoundError(name)

TRAIN_DIR = find_dir(path, "Training")
TEST_DIR  = find_dir(path, "Testing")
print(TRAIN_DIR)
print(TEST_DIR)
for d in sorted(os.listdir(TRAIN_DIR)):
    print(d, len(os.listdir(os.path.join(TRAIN_DIR, d))), "صورة تدريب")

## 2) تجهيز الداتا

In [ ]:
import json
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers

IMG_SIZE = (224, 224)
BATCH = 32
SEED = 42

train_ds = keras.utils.image_dataset_from_directory(
    TRAIN_DIR, validation_split=0.15, subset="training", seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH, label_mode="int")
val_ds = keras.utils.image_dataset_from_directory(
    TRAIN_DIR, validation_split=0.15, subset="validation", seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH, label_mode="int")
test_ds = keras.utils.image_dataset_from_directory(
    TEST_DIR, image_size=IMG_SIZE, batch_size=BATCH, label_mode="int", shuffle=False)

class_names = train_ds.class_names
print("Classes:", class_names)
assert len(class_names) == 4, "المفروض 4 تصنيفات بس"

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

## 3) بناء الموديل (MobileNetV2 + Transfer Learning)

In [ ]:
data_aug = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name="augmentation")

base = keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet")
base.trainable = False

inputs = keras.Input(shape=IMG_SIZE + (3,))
x = data_aug(inputs)
x = layers.Rescaling(1 / 127.5, offset=-1)(x)   # التحجيم جوه الموديل نفسه
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(class_names), activation="softmax")(x)
model = keras.Model(inputs, outputs)

model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

## 4) التدريب — المرحلة الأولى (تدريب الطبقة الأخيرة فقط)

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
]
history1 = model.fit(train_ds, validation_data=val_ds, epochs=8, callbacks=callbacks)

## 5) التدريب — المرحلة التانية (Fine-tuning لتحسين الدقة)

In [ ]:
base.trainable = True
for layer in base.layers[:-40]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(1e-5),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history2 = model.fit(train_ds, validation_data=val_ds, epochs=6, callbacks=callbacks)

## 6) تقييم الموديل على صور الاختبار

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"Test accuracy: {test_acc * 100:.2f}%")

y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_pred = np.argmax(model.predict(test_ds, verbose=0), axis=1)

report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
print(classification_report(y_true, y_pred, target_names=class_names))

# منحنيات التدريب
acc = history1.history["accuracy"] + history2.history["accuracy"]
val_acc = history1.history["val_accuracy"] + history2.history["val_accuracy"]
loss = history1.history["loss"] + history2.history["loss"]
val_loss = history1.history["val_loss"] + history2.history["val_loss"]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(acc, label="train"); ax[0].plot(val_acc, label="validation")
ax[0].set_title("Accuracy"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(loss, label="train"); ax[1].plot(val_loss, label="validation")
ax[1].set_title("Loss"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout(); plt.savefig("training_curves.png", dpi=150); plt.show()

# مصفوفة الالتباس
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
plt.tight_layout(); plt.savefig("confusion_matrix.png", dpi=150); plt.show()

## 7) حفظ الموديل وتنزيل الملفات

In [ ]:
import shutil
from google.colab import files

model.save("brain_tumor_model.keras")
with open("class_names.json", "w", encoding="utf-8") as f:
    json.dump(class_names, f)

results = {
    "test_accuracy": float(test_acc),
    "tensorflow_version": tf.__version__,
    "classification_report": report,
}
with open("results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

import zipfile
with zipfile.ZipFile("brain_tumor_results.zip", "w") as z:
    for name in ["brain_tumor_model.keras", "class_names.json", "results.json",
                 "training_curves.png", "confusion_matrix.png"]:
        z.write(name)

print(f"✅ خلص! دقة الموديل على الاختبار: {test_acc * 100:.2f}%")
files.download("brain_tumor_results.zip")